In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

In [2]:
pos_folder = r"C:\Git_Repo\final-capstone-e2ws-ai-topia-consulting\data\raw\pothole_images\positive"
neg_folder = r"C:\Git_Repo\final-capstone-e2ws-ai-topia-consulting\data\raw\pothole_images\negative"

def build_dataset(folder, label):
    rows = []
    for f in os.listdir(folder):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            rows.append({
                "filepath": os.path.join(folder, f),
                "label": label
            })
    return rows

df = pd.DataFrame(build_dataset(pos_folder, 1) + build_dataset(neg_folder, 0))
print(df["label"].value_counts())

label
0    2259
1    1485
Name: count, dtype: int64


In [3]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print(len(train_df), len(val_df), len(test_df))

2620 562 562


In [4]:
IMG_SIZE = 384
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

def crop_road_region(img):
    h = tf.shape(img)[0]
    w = tf.shape(img)[1]

    y1 = tf.cast(tf.cast(h, tf.float32) * 0.30, tf.int32)
    y2 = tf.cast(tf.cast(h, tf.float32) * 0.78, tf.int32)
    x1 = tf.cast(tf.cast(w, tf.float32) * 0.05, tf.int32)
    x2 = tf.cast(tf.cast(w, tf.float32) * 0.95, tf.int32)

    return img[y1:y2, x1:x2, :]

In [5]:
data_augmentation = tf.keras.Sequential([
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.20),
    layers.RandomBrightness(0.15),
    layers.GaussianNoise(0.03),
], name="augmentation")

In [6]:
def make_load_fn(preprocess_fn):
    def load_image(filepath, label):
        img = tf.io.read_file(filepath)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = crop_road_region(img)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = preprocess_fn(img * 255.0)
        return img, tf.cast(label, tf.float32)
    return load_image


def make_dataset(dataframe, preprocess_fn, training=False):
    load_fn = make_load_fn(preprocess_fn)
    ds = tf.data.Dataset.from_tensor_slices(
        (dataframe["filepath"].values, dataframe["label"].values)
    )
    ds = ds.map(load_fn, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(256, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


In [7]:
class_weight = {0: 1.0, 1: 2.0}

In [8]:
def build_transfer_model(backbone_name, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    if backbone_name == "MobileNetV2":
        base_model = MobileNetV2(
            input_shape=input_shape,
            include_top=False,
            weights="imagenet"
        )
    elif backbone_name == "EfficientNetB0":
        base_model = EfficientNetB0(
            input_shape=input_shape,
            include_top=False,
            weights="imagenet"
        )
    elif backbone_name == "ResNet50":
        base_model = ResNet50(
            input_shape=input_shape,
            include_top=False,
            weights="imagenet"
        )
    else:
        raise ValueError("Unsupported backbone")

    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )
    return model, base_model

In [9]:
def train_model(model, base_model, train_ds, val_ds, class_weight, fine_tune_layers=30, name="model"):
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            f"{name}_best.keras",
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        )
    ]

    print(f"\nTraining head: {name}")
    history1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=15,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=1
    )

    print(f"\nFine-tuning: {name}")
    base_model.trainable = True
    for layer in base_model.layers[:-fine_tune_layers]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )

    history2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=1
    )

    return history1, history2

In [10]:


train_ds_effnet = make_dataset(train_df, efficientnet_preprocess, training=True)
val_ds_effnet   = make_dataset(val_df, efficientnet_preprocess, training=False)
test_ds_effnet  = make_dataset(test_df, efficientnet_preprocess, training=False)



In [11]:

effnet_model, effnet_base = build_transfer_model("EfficientNetB0")




train_model(
    effnet_model, effnet_base,
    train_ds_effnet, val_ds_effnet,
    class_weight,
    name="efficientnetb0"
)




Training head: efficientnetb0
Epoch 1/15
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 519ms/step - accuracy: 0.4790 - loss: 0.9939 - precision: 0.4036 - recall: 0.6376
Epoch 1: val_loss improved from None to 0.70220, saving model to efficientnetb0_best.keras

Epoch 1: finished saving model to efficientnetb0_best.keras
164/164 ━━━━━━━━━━━━━━━━━━━━ 121s 649ms/step - accuracy: 0.4702 - loss: 0.9882 - precision: 0.4006 - recall: 0.6766 - val_accuracy: 0.4804 - val_loss: 0.7022 - val_precision: 0.4274 - val_recall: 0.9103 - learning_rate: 1.0000e-04
Epoch 2/15
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 555ms/step - accuracy: 0.5151 - loss: 0.9331 - precision: 0.4374 - recall: 0.7848
Epoch 2: val_loss improved from 0.70220 to 0.67379, saving model to efficientnetb0_best.keras

Epoch 2: finished saving model to efficientnetb0_best.keras
164/164 ━━━━━━━━━━━━━━━━━━━━ 112s 659ms/step - accuracy: 0.5324 - loss: 0.9234 - precision: 0.4490 - recall: 0.7873 - val_accuracy: 0.5391 - val_loss: 0.6738 - val_precision: 0.4605

(<keras.src.callbacks.history.History at 0x2a5c5af7cb0>,
 <keras.src.callbacks.history.History at 0x2a5ca208190>)

In [12]:

val_probs_effnet = effnet_model.predict(val_ds_effnet).ravel()
val_true = val_df["label"].values

36/36 ━━━━━━━━━━━━━━━━━━━━ 20s 510ms/step


In [13]:
val_probs_ensemble = val_probs_effnet 


In [14]:
thresholds = np.arange(0.20, 0.81, 0.02)
results = []

for t in thresholds:
    val_pred = (val_probs_ensemble >= t).astype(int)
    results.append({
        "threshold": t,
        "accuracy": accuracy_score(val_true, val_pred),
        "weighted_f1": f1_score(val_true, val_pred, average="weighted")
    })

threshold_df = pd.DataFrame(results).sort_values(
    ["weighted_f1", "accuracy"],
    ascending=False
)

print(threshold_df.head(10))

best_threshold = threshold_df.iloc[0]["threshold"]
print("Best ensemble threshold:", best_threshold)

    threshold  accuracy  weighted_f1
19       0.58  0.857651     0.857540
20       0.60  0.857651     0.856921
18       0.56  0.848754     0.849233
21       0.62  0.850534     0.848492
23       0.66  0.850534     0.847124
25       0.70  0.850534     0.844993
16       0.52  0.841637     0.843089
22       0.64  0.845196     0.842404
24       0.68  0.845196     0.840859
26       0.72  0.846975     0.840473
Best ensemble threshold: 0.5799999999999998


In [15]:

test_probs_effnet = effnet_model.predict(test_ds_effnet).ravel()

test_true = test_df["label"].values

36/36 ━━━━━━━━━━━━━━━━━━━━ 17s 454ms/step


In [16]:
test_probs_ensemble = test_probs_effnet 


In [17]:
test_pred_ensemble = (test_probs_ensemble >= best_threshold).astype(int)

In [18]:
print("Ensemble Classification Report:\n")
print(classification_report(
    test_true,
    test_pred_ensemble,
    target_names=["no_pothole", "pothole"]
))

cm = confusion_matrix(test_true, test_pred_ensemble)
print("Ensemble Confusion Matrix:\n", cm)

acc = accuracy_score(test_true, test_pred_ensemble)
wf1 = f1_score(test_true, test_pred_ensemble, average="weighted")

print(f"Ensemble Accuracy: {acc:.4f}")
print(f"Ensemble Weighted F1: {wf1:.4f}")

Ensemble Classification Report:

              precision    recall  f1-score   support

  no_pothole       0.89      0.86      0.88       339
     pothole       0.80      0.83      0.82       223

    accuracy                           0.85       562
   macro avg       0.84      0.85      0.85       562
weighted avg       0.85      0.85      0.85       562

Ensemble Confusion Matrix:
 [[293  46]
 [ 37 186]]
Ensemble Accuracy: 0.8523
Ensemble Weighted F1: 0.8528
